# MiniLAr + CNN

In this notebook, we design and train a Convolutional Neural Network (CNN) for the same MiniLAr particle-image classification task from the previous notebook. MiniLAr is MNIST-shaped, but the images are tiny LArTPC particle projections rather than handwritten digits.

## Goals
1. Design a CNN and train it on MiniLAr
2. Compare a CNN to the previous MLP
3. Think about why image-local structure matters

Let's start with usual import!

In [ ]:
# Import drawing tools, set style
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

mpl.rcParams['figure.figsize'] = [8, 6]
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.grid'] = True

# Import numpy and torch, set default device based on GPU availability
import numpy as np
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Set the seed to always get the same result from RNG calls in this notebook
SEED = 12345
np.random.seed(SEED)    # Setting the seed for reproducibility
torch.manual_seed(SEED) # This is how you do for torch!

## Create MiniLAr Dataset
Following the previous notebook, let's create train and test datasets and dataloaders.

In [ ]:
from dataset.lartpc_particles import LArTPCParticles

# Specify where you want the dataset to land
LOCAL_DATA_DIR = 'dataset/minilar-dataset'

# Fill this in once the dataset file is hosted.
# You can also use a local path, e.g. DATA_URL='/path/to/lartpc_particles.pt'
DATA_URL = 'https://s3df.slac.stanford.edu/data/neutrino/dune-tech-analysis/mlar/raw/lartpc_particles.pt'

# Use prepared data handler from pytorch (torchvision)
train_dataset = LArTPCParticles(
    LOCAL_DATA_DIR,
    train=True,
    download=DATA_URL is not None,
    url=DATA_URL,
)

train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=64,
                                           shuffle=True,
                                           num_workers=4)

# Use prepared data handler from pytorch (torchvision)
test_dataset = LArTPCParticles(
    LOCAL_DATA_DIR,
    train=False,
    download=DATA_URL is not None,
    url=DATA_URL,
)

test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=64,
                                          shuffle=False,
                                          num_workers=4)

### Prompt

Before looking at the CNN, what local image patterns might help distinguish particle classes: straight tracks, diffuse showers, track length, local brightness, or gaps/scattering?

### Define train and test functions

We recycle those from the previous MLP notebook!

In [ ]:
# Python time tracking package
import time

# Python CSV writing package
import csv

# Progress bar package, makes waiting easier!
# If you're curious, tqdm comes from the arabic word for progress, taqadum
from tqdm import tqdm

def run_train(model, loader,  
              num_iterations=100, log_dir='logs_mlp', log_prefix=None,
              optimizer='SGD', lr=0.001, device=None):
    """Train loop function.
    
    Parameters
    ----------
    model : torch.nn.Module
        Machine learning model
    loader : torch.DataLoader
        Batch loading tool
    num_iterations : int, default 100
        Number of iterations to run the optimization over
    log_dir : str, default 'logs_mlp'
        Path to the log directory
    log_prefix : str, optional
        Prefix to prepend onto log names
    optimizer : str, default 'SGD'
        Name of the optimizer as defined here: https://docs.pytorch.org/docs/stable/optim.html
    lr : float, default 0.001
        Learning rate
    """
    # Create the log directory if it does not exist
    !mkdir -p {log_dir}

    # Define log name
    prefix = '' if log_prefix is None else f'{log_prefix}_'
    train_log_name = f'{log_dir}/{prefix}train.csv'

    # Initialize the loss function (cross-entropy loss)
    criterion = torch.nn.CrossEntropyLoss()

    # Initialize the optimizer
    optimizer_fn = getattr(torch.optim,optimizer)
    optimizer = optimizer_fn(model.parameters(), lr=lr)

    # Initialize a progress bar
    pbar = tqdm(total=num_iterations, position=0, leave=True)

    # Loop over the entire dataset until the requested number of iterations is reached
    iteration = 0
    with open(train_log_name, 'w') as csvfile:
        # Initialize CSV writer
        writer = csv.writer(csvfile)
        writer.writerow(['iter', 'epoch', 'loss'])
        while iteration < num_iterations:
            # Loop over data in the loader (one epoch)
            for data, label in loader:
                # If a device is specify, move data/label to it
                if device is not None:
                    data, label = data.to(device), label.to(device)

                optimizer.zero_grad()
    
                loss = criterion(model(data), label)
                
                loss.backward()
                optimizer.step()
                
                # Append the log
                writer.writerow([iteration, iteration/len(loader), loss.item()])
    
                # Increment iteration count, update the progress bar
                iteration += 1
                pbar.update(1)

                # Break if the necessary number of iterations is reached
                if iteration >= num_iterations:
                    break


def run_test(model, loader, device=None):
    """Runs the trained model on test data.

    Parameters
    ----------
    model : torch.nn.Module
        Model definition
    loader : torch.DataLoader
        Method to load the test data
    device : str, optional
        Device on which to put the data/model if one uses a GPU

    Returns
    -------
    torch.Tensor
        (N) List of labels for each image in the test set
    torch.Tensor
        (N, 5) Softmax scores predicted for each of the 5 particle classes
    """
    # Initialize the output lists
    label_v, softmax_v = [],[]

    # Initialize the softmax function, s_i = exp(-v_i)/(sum_i exp(-v_i))
    softmax   = torch.nn.Softmax(dim=1)

    # Initialize a progress bar
    pbar = tqdm(total=len(loader), position=0, leave=True)

    # Loop over test data, append labels and softmax predictions
    with torch.set_grad_enabled(False):
        for data, label in loader:
            if device is not None:
                data,label = data.to(device), label.to(device)
            label_v.append  ( label.detach().reshape(-1)   )
            softmax_v.append( softmax(model(data)).detach())
            pbar.update(1)

    # Return
    return torch.concat(label_v).cpu().numpy(), torch.concat(softmax_v).cpu().numpy()

## Particle classification with CNN

We now design a CNN for the same MiniLAr task. A CNN is better matched to images because it learns local spatial filters before making a classification decision. Let's define 3 convolution layers followed by LeakyReLU activations and MaxPool2d downsampling.

### Prompt

What local patterns might matter in these LArTPC particle images: straight tracks, showers, track length, local brightness, topology?

In [ ]:
NUM_CLASSES = 5

class CNN(torch.nn.Module):
    """Basic convolution neural network (CNN) model."""

    def __init__(self, num_filters=16):
        """Initialize the CNN.

        Parameters
        ----------
        num_filters : int, default 16
            Number of filter kernels (convolution matrices)
        """
        # Initialize the parent class
        super().__init__()
        
        # CNN feature extractor definition
        self._feature_extractor = torch.nn.Sequential(
            torch.nn.Conv2d(1, num_filters, 3, padding=1),             # 1 to N_f conv, kernel size 3, padding 1 (preserve size)
            torch.nn.LeakyReLU(),                                      # LeakyReLU activation
            torch.nn.MaxPool2d(2, 2),                                  # Max pooling, now image is 14*14
            torch.nn.Conv2d(num_filters, num_filters*2, 3, padding=1), # N_f to 2*N_f conv, kernel size 3, padding 1
            torch.nn.LeakyReLU(),                                      # LeakyReLU activation
            torch.nn.MaxPool2d(2, 2),                                  # Max pooling, now image is 7*7
            torch.nn.Conv2d(num_filters*2,num_filters*4,3,padding=1),  # 2*N_f to 4*N_f conv, kernel size 3, padding 1
            torch.nn.LeakyReLU(),                                      # LeakyReLU activation
            torch.nn.MaxPool2d(7, 7))                                  # Final max pooling, pools everything into a single feature vector
        
        # Classifier MLP, converts the feature vector (4*N_f) into a 10 values (one per possible class)
        self._classifier = torch.nn.Linear(num_filters*4, NUM_CLASSES)

    def forward(self, x):
        """Pass one batch of data through the model.

        Parameters
        ----------
        x : torch.Tensor
            (N, 1, 28, 28) N MiniLAr images of size 28*28

        Returns
        -------
        torch.Tensor
            (N) Logit output of the classifier layer (one number per image)
        """
        
        # Extract features
        features = self._feature_extractor(x)
        
        # Flatten the 3d tensor (2d space x channels = features)
        features = features.view(-1, np.prod(features.size()[1:]))
        
        # Classify and return
        return self._classifier(features)

### Prompt

Follow the feature-map sizes through the model. The image starts as `[1, 28, 28]`. What happens after each `MaxPool2d(2, 2)` layer? Why does downsampling help?

### Train the model
Let's use the train loop to train the model.

In [ ]:
# Use the device selected at the top of the notebook.
print('Training on device:', device)

# Initialize a CNN with 16 filters (place on the correct device!)
model = CNN(16).to(device=device)

# Run the train loop
run_train(model, train_loader, 10000, log_dir='logs_cnn', optimizer='Adam', device=device)

If your machine is slow, skip over this training process and simply load the model parameters from an earlier training process!

The way this file was produced was by simply calling
```python
torch.save(model.state_dict(), 'weights/cnn_model.ckpt')
```

In [ ]:
# torch.save(model.state_dict(), 'weights/cnn_model.ckpt')

In [ ]:
# state_dict = torch.load('weights/cnn_model.ckpt', weights_only=True)
# model.load_state_dict(state_dict)

Now we look at the train log, we use the one trained earlier to save time...

In [ ]:
import pandas as pd

df = pd.read_csv('logs_cnn/train.csv')
# df = pd.read_csv('logs_cnn/pretrain.csv')
df

In [ ]:
df.rolling(10).mean().plot('epoch', 'loss', grid=True)

You can see that this model converges to a lower loss value than the MLP.

Before checking the number: do you expect the CNN to help more for MiniLAr than it would for a table of unrelated features? Why?

Now let us evaluate exactly how much better it does on a test set...

### Prompt

The MLP saw the same pixels after flattening, but the CNN keeps nearby pixels together. What information does the CNN preserve that the MLP mostly discards?

In [ ]:
labels, scores = run_test(model, test_loader, device=device)

In [ ]:
# torch.save({'labels': labels, 'scores': scores}, 'inference/cnn_output.pkl')

Once again, if your machine is slow, I have done this for you upstream, simply load the information!

In [ ]:
# saved_data = torch.load('inference/cnn_output.pkl', weights_only=False)
# labels, scores = saved_data['labels'], saved_data['scores']

Now let's check accuracy:

In [ ]:
preds = np.argmax(scores, axis=1)
np.sum(labels == preds)/len(labels)

The CNN improves over the MLP as local image structure is useful for this task.

A confusion matrix gives more detail:

In [ ]:
# Define a function which estimates the confusion matrix of the model.
def make_confusion_matrix(labels, preds):
    """Make a row-normalized confusion matrix."""
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=float)
    for label, pred in zip(labels, preds):
        cm[int(label), int(pred)] += 1

    row_sums = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums > 0)


def plot_confusion_matrix(cm, class_names):
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm, vmin=0, vmax=1, cmap='viridis')
    fig.colorbar(image, ax=ax, label='fraction of true class')
    ax.grid(False)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Prediction')
    ax.set_ylabel('True label')

    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            value = cm[row, col]
            color = 'black' if value > 0.5 else 'white'
            ax.text(col, row, f'{value:.2f}', ha='center', va='center', color=color)

    fig.tight_layout()
    return fig, ax


cm = make_confusion_matrix(labels, preds)
plot_confusion_matrix(cm, train_dataset.class_names);

### Exercise 1

Train the CNN model on GPU and compare the wall-time to CPU training.

In [ ]:
# Your code here!

You should see that, for CNNs, GPU gives a substantial speed-up. This is because CNNs apply many filter operations over local image patches, and those operations benefit from parallelization.

### Exercise 2
How many parameters are there in our CNN model? How does that compare to the MLP from the previous notebook?

In [ ]:
# Your code here!